In [ ]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.visualization import plot_histogram
from qiskit.result import marginal_counts
from qiskit.circuit import ControlledGate
from qiskit.circuit.classical import expr
import math

In [ ]:
# This exercise is to implement a cryptographic protocol called "quantum secret sharing".

# Alice has a qubit that she wants to transfer to either Bob or Carol, but she requires
# both of them to collaborate in order to complete the transfer.

# It's a similar idea to teleportation, but the requirement for collaboration is
# supposed to guard against a dishonest participant.

# The idea comes from work by Hillery, Buzek and Berthiaume in 1998.
# Their article has some nice motivation: https://arxiv.org/pdf/quant-ph/9806063

# This exercise is based on the protocol in Section 3 of the article.

# Alice constructs a GHZ state on three qubits:

# a, b, c = 1/sqrt(2) ( |000> + |111> )

# Alice also has a qubit x, in an arbitrary state, that she wants to transfer to
# either Bob or Carol.

# Alice entangles x with the GHZ state, not only by CNOT(x,a) as in teleportation,
# but by CNOT(x,a) then CNOT(x,b) then CNOT(x,c).

# Alice now sends b to Bob and c to Carol (we can't represent this in a Python
# implementation because we are just working with a sequential program).

# Now suppose Alice wants to transfer x to Carol, requiring collaboration from Bob.

# Alice measures x and a in the Bell basis (explained below),
# obtaining a 2-bit result consisting of r_x and r_a.

# Alice sends r_x and r_a to Carol (this is classical communication).

# Bob measures b in the diagonal basis, obtaining result r_b. He sends r_b to Carol.

# Carol now applies operations to c, depending on the values of r_x, r_a, r_b:

# r_x  r_a  r_b     operations
# ----------------------------
#  0    0    0         none
#  0    0    1         Z(c)
#  0    1    0         Z(c)
#  0    1    1         none
#  1    0    0         X(c)
#  1    0    1    Z(c) then X(c)
#  1    1    0    Z(c) then X(c)
#  1    1    1         X(c)

# At the end of all this, the state of c should be the original state of x.

# Now to explain what "measure in the Bell basis" means.

# The following four 2-qubit states form a basis, meaning that any 2-qubit
# state can be expressed as a linear combination of these four states. This
# is called the Bell basis. (The first state is the one that is often
# referred to as "the Bell state".)

# 1/sqrt(2) ( |00> + |11> )
# 1/sqrt(2) ( |00> - |11> )
# 1/sqrt(2) ( |01> + |10> )
# 1/sqrt(2) ( |01> - |10> )

# To measure in this basis, we apply operations so that these four states are
# converted into the standard basis states: |00>, |01>, |10>, |11>
# and then do a standard basis measurement.

# We just have to be careful about how the 2-bit result of the measurement is
# interpreted. That's the same thing as being clear about the order in which we
# write the Bell basis states.

# To make this work in Qiskit for the secret sharing protocol, we should apply
# cx(1,0) and then h(1).

# We can check what happens to the Bell states above (numbering the qubits as
# 0 on the left, 1 on the right):

# State                           After cx(1,0)                  After h(1)
# 1/sqrt(2) ( |00> + |11> )       1/sqrt(2) ( |00> + |10> )      |00>
# 1/sqrt(2) ( |00> - |11> )       1/sqrt(2) ( |00> - |01> )      |01>
# 1/sqrt(2) ( |01> + |10> )       1/sqrt(2) ( |11> + |10> )      |10>
# 1/sqrt(2) ( |01> - |10> )       1/sqrt(2) ( |01> - |11> )      |11>

# Numbering the qubits in Qiskit can be confusing, but if you number
# x,a,b,c  as  0,1,2,3 and use cx(1,0) followed by h(1) for the Bell basis
# measurement, everything should work.



# Implement the protocol and test it. If you think of it as an extension of
# teleportation, you can get a long way by starting with Lab2-B. To do the
# operations at the end, conditionally on the results of the measurement,
# you can adapt the bit-flip code implementation from Lab4-A.

